# Fact Data Modeling: Core Concepts, Deduplication Day 1 Lecture

A fact is something that happened or occurred.
- A user logs in to an app
- A transaction is made

Think of facts as something that can't breakdown any further. Think of components of an atom (protons, electrons, neutrons). These can't break down any further.

Facts are more challenging than dimensions since you're working with a lot more data.

There's a lot more prevalence of duplicates in fact data versus dimension data. Two examples of how duplicates can occur:
- Software engs cause bug that results in actions being logged twice in logs
- If we were facebook and pushed a notification to someone and then they clicked on the notification and link and then came back later and clicked on it again, this would result in two clicks. This isn't a bug, but if you were to calculate click thru rate, you'd show 200%. Therefore, you'd need to know how to handle that properly.

You can have normalized fact vs denormalized fact.
- Normalized facts dont have any dimensional attributes. Just ids to join to get that info.
- Denormalized facts bring in some dimensional attributes for quicker analysis at the cost of more storage.

Both have their place in the world.

Raw logs <> Fact data

Raw logs have ugly schemas designed for online systems. Potentially contain dups, and have shorter retention.

Fact data have nice column names, have quality guarantees, longer retention.

When should you bring in dimensional data in fact data?

Pretty much when you have an extremely large scale problem. I do not think this should be used except for very few uses cases. Zach gave an example while working at netflix where they were trying to track which apps talk to each other. However, there's petabytes of data a day thats generated and all they have are ip addresses. And so they had to join a big datatable onto anther and that simply didn't work. So he went upstream had got software engs to start logging the name of the app in the logs so they dont have to do joins. However, issue here is that if there are changes in the name then you get issues.

Other potential fixes when working with high volumne data
- Sampling (think a/b testing where you can take samples)
- Bucketing
    - (i think he means partitioning, which we do with dates in my professional work)

How to remove duplicates?
- Streaming to deduplicate facts is an option. Most duplicates occur shortly after the initial event (15 min to 1 hour). 

He discussed his solution for deduping facebook notifications. I think he talked about how it took 9 hours to dedeup facebook notifications, but they wanted to reduce the time for this. His solution was essentially to batch it up. He'd dedupe within an hour, then across hours, and then so on and so forth. At the end though, the two tables combined to create a final dedeuped data within intraday.

This was an interesting process and i'm surprised it worked, especially since joining the two tables at the end i'd think were big too. But I think this goes to show when it comes to data eng, you gotta think outside the box. The original issue was related to memory and size of table. But by chunking it up, he was able to do it within an hour.

This honestly seems like a messy solution, but I think sometimes messy solutions are needed.

## Lab

Key insight on fact data modeling: if you can join a dimension cheaply, then it should be normalized.

I found it interesting at minute 20, we create new columns from a raw string column. Makes a lot of sense. So the column was "comment" and it had raw strings. But we created some new boolean columns that matched on certain parts of the string. Very cool

# Fact Data Modeling: Core Elements in Data Modeling Day 2 Lecture

Sometimes dimensions can be an aggregation of facts. At facebook, there was a dim_is_active dimension that based on fact data! Sometimes its helpful to create dimensions from an aggregation of facts.

One general statement: dimensions are typically used to group by. They also are typically derived from a snapshot of state. Facts are aggregated. Facts come from events or logs.

A good example of blurry lines is price on an airbnb. Is price a dimension or fact? You can run an average on price to get average price. However, price is an attribute of the listing and night. It's kind of inbetween, but leans more on the dimension side since its an attribute of the night.

Side note: seems like dimensions that are aggregation of facts are a good way to incorporate fact data at the tables grain level. Thinking back to the facebook example of dim_is_active.

## Lab

One thing I noticed him using in the lab is generate_series(). There's definitely a lot of functions in postgres sql that I didn't know.

The second half of this lab was pretty wild. He showed how to create a binary number to represent activity over the last 30 days or so. I would have figured it would have been a date array, but I can see how this is extremely efficient.

# Fact Data Modeling: Minimizing Shuffle and Reducing Facts Day 3 Lecture

Shuffle is the bottleneck to parrelization

He talks about fact data and strategies to reduce the size of fact data. One way is to aggregate to daily or change data structure to reduce rows. However, this comes with a trade off since this may hinder your ability to do various analytics.

## Lab